# Grain boundary module: `grainboundary.py` 

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('../../..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import gbtk.grainboundary as grainboundary
import numpy as np
import pandas as pd

## Constructing a CSL grain boundary 

We now want to build ourselves a grain boundary object, perhaps to do some simulation work.

The first step is to instantiate a grain boundary object with a suitable lattice type. We'll switch on the debug output so that we can better see what is going on:

In [2]:
test_gb = grainboundary.GrainBoundary('fcc')
test_gb.set_debug()

A CSL grain boundary will be built from CSL unit cells. a `grainboundar` object contains a `csl` object and to begin we need to set the axis and angle associated with the CSL grain boundary (noting that we might have used the csl cataloguing functions to find appropriate values for `m` and `n`).

In [3]:
h = 1; k = 1; l = 1
m = 1; n = 5
test_gb.set_axis([h,k,l])
test_gb.set_angle_mn(m,n)

Next we need to calculate the CSL basis. This is done with a call to the method:

`set_csl(self,csl_vectors_black=None,csl_vectors_white=None)`

By default the CSL basis will be calculated automatically, but it is possible to use a shortcut by passing optional arrays containing the CSL lattice vectors. This may save time in the case of very large $\Sigma$ boundaries whose basis vectors have already been recorded in a catalogue. Here we make the default call:

In [4]:
test_gb.set_csl()

----------------------------------------------------
csl.find_csl_basis() debug:
***************************
CSL Cell vectors (lattice basis)
Black                      White
[      1      1      1 ]   [      1      1      1 ]
[      0      1     -2 ]   [      1      0     -2 ]
[     -1      2      0 ]   [      0      2     -1 ]
CSL Cell vectors (cartesian basis)
Black                                       White
[     1.000000     1.000000     1.000000]   [     1.000000     1.000000     1.000000 ]
[     0.000000     1.000000    -2.000000]   [     1.000000     0.000000    -2.000000 ]
[    -1.000000     2.000000     0.000000]   [     0.000000     2.000000    -1.000000 ]



You will recognise this output, as `set_csl()` has called the `csl.find_csl_basis()` function.

The next step is to specify a boundary plane via three integers which specify the components of the grain boundary normal in the csl basis. This determines the minimum repeat unit of crystal in which the chosen boundary can be created. The debug output contains useful information about this grain boundary cell and the number of atoms it contains.

In [11]:
H = 1; K = 1; L = 0
#H = 1; K = 0; L = 0
#H = 0; K = 1; L = 1
test_gb.set_boundary_plane_csl([H,K,L], search_radius=20)

----------------------------------------------------
grainboundary.set_boundary_plane_csl() debug:
*********************************************
Grain boundary normal (lattice basis)
[      1      2     -1 ]
First vector (lattice basis)
[     -3      2      1 ]
Second vector (lattice basis)
[      2      1      4 ]
Grain boundary cell vectors (csl basis)
[     -1     -1      2 ]
[      2     -1      0 ]
[      0      0     -1 ]
Grain boundary cell vectors (lattice basis)
Black                      White
[     -3      2      1 ]   [     -2      3     -1 ]
[      2      1      4 ]   [      1      2      4 ]
[      1     -2      0 ]   [      0     -2      1 ]
Grain boundary cell vectors (cartesian basis)
Black                                       White
[    -3.000000     2.000000     1.000000]   [    -2.000000     3.000000    -1.000000 ]
[     2.000000     1.000000     4.000000]   [     1.000000     2.000000     4.000000 ]
[     1.000000    -2.000000     0.000000]   [     0.000000    -2.

True

We get plenty of useful information in the debug output, including the full specification of the grain boundary cell and the number of csl unit cells, lattice unit cells and atoms that it contains. The debug also informs us if the boundary is a twist, tilt, symmetric tilt or twist boundary.

In general, the grain boundary cell will contain multiple copies of the CSL unit cell and can become quite large, depending on the selected grain boundary plane. To help with understanding how the values of `H`, `K` and `L` affect the grain boundary plane and type of boundary, there is cataloguing functionality built into the `grainboundary` module that we explore shortly.

But first we will take a look at the grain boundary that we have generated.


## Visualising the grain boundary cell
There are two ways to visualise the grain boundary cell. First, in the original orientation in the crystal lattice. The grain boundary cells are shown, but also the CSL unit cells for comparison and the boundary normals:

In [14]:
test_gb.visualise_3d()

----------------------------------------------------
Grain boundary visualisation:
*****************************



And second, after the specified CSL rotation, which should bring the black and white cells into coincidence.

In [15]:
test_gb.visualise_3d_rotated()

----------------------------------------------------
Grain boundary visualisation:
*****************************



## Creating a grain boundary catalogue
The `grainboundary.py` module also contains functionality to catalogue possible grain boundary planes for a given CSL axis-angle combination. The usage is very similar to the CSL cataloguing functions and an example is given below. The output includes the grain boundary plane normals specified in the crystal lattice basis in each half of the bicrystal. 

These catalogues can take quite a while to build, especially for large values of `limit`, but the one below should only take around 5 minutes to produce.

In [16]:
h = 1; k = 1; l = 1
m = 1; n = 5
limit = 3
grainboundary.write_gb_catalogue('catalogues/gb/', 'fcc', [h,k,l], m, n, limit, search_radius=20)

,h,k,l,m,n,theta,sigma,H,K,L,num_atoms,type,N_b[0],N_b[1],N_b[2],N_w[0],N_w[1],N_w[2]
0,1,1,1,1,5,38.213211,7.0,1.0,-2.0,2.0,28.0,mixed,-1.0,3.0,5.0,-1.0,5.0,3.0
1,1,1,1,1,5,38.213211,7.0,-2.0,1.0,0.0,28.0,mixed,-2.0,-1.0,-4.0,-1.0,-2.0,-4.0
2,1,1,1,1,5,38.213211,7.0,-1.0,-2.0,1.0,28.0,symmetric,-2.0,-1.0,3.0,-3.0,1.0,2.0
3,1,1,1,1,5,38.213211,7.0,-1.0,-1.0,2.0,28.0,symmetric,-3.0,2.0,1.0,-2.0,3.0,-1.0
4,1,1,1,1,5,38.213211,7.0,-2.0,0.0,-1.0,28.0,mixed,-1.0,-4.0,-2.0,-2.0,-4.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,1,1,1,1,5,38.213211,7.0,1.0,2.0,3.0,560.0,mixed,-2.0,9.0,-3.0,3.0,7.0,-6.0
286,1,1,1,1,5,38.213211,7.0,-1.0,-3.0,-3.0,616.0,mixed,2.0,-10.0,5.0,-4.0,-7.0,8.0
287,1,1,1,1,5,38.213211,7.0,1.0,3.0,3.0,616.0,mixed,-2.0,10.0,-5.0,4.0,7.0,-8.0
288,1,1,1,1,5,38.213211,7.0,2.0,3.0,3.0,644.0,mixed,-1.0,11.0,-4.0,5.0,8.0,-7.0


As before, we can take a look at the output by reading it back into a pandas dataframe

In [20]:
gb_df = pd.read_hdf('catalogues/gb/gb_cat_fcc_1_1_1__1_5_limit_3.df','df')
gb_df

,h,k,l,m,n,theta,sigma,H,K,L,num_atoms,type,N_b[0],N_b[1],N_b[2],N_w[0],N_w[1],N_w[2]
0,1,1,1,1,5,38.213211,7.0,1.0,-2.0,2.0,28.0,mixed,-1.0,3.0,5.0,-1.0,5.0,3.0
1,1,1,1,1,5,38.213211,7.0,-2.0,1.0,0.0,28.0,mixed,-2.0,-1.0,-4.0,-1.0,-2.0,-4.0
2,1,1,1,1,5,38.213211,7.0,-1.0,-2.0,1.0,28.0,symmetric,-2.0,-1.0,3.0,-3.0,1.0,2.0
3,1,1,1,1,5,38.213211,7.0,-1.0,-1.0,2.0,28.0,symmetric,-3.0,2.0,1.0,-2.0,3.0,-1.0
4,1,1,1,1,5,38.213211,7.0,-2.0,0.0,-1.0,28.0,mixed,-1.0,-4.0,-2.0,-2.0,-4.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,1,1,1,1,5,38.213211,7.0,1.0,2.0,3.0,560.0,mixed,-2.0,9.0,-3.0,3.0,7.0,-6.0
286,1,1,1,1,5,38.213211,7.0,-1.0,-3.0,-3.0,616.0,mixed,2.0,-10.0,5.0,-4.0,-7.0,8.0
287,1,1,1,1,5,38.213211,7.0,1.0,3.0,3.0,616.0,mixed,-2.0,10.0,-5.0,4.0,7.0,-8.0
288,1,1,1,1,5,38.213211,7.0,2.0,3.0,3.0,644.0,mixed,-1.0,11.0,-4.0,5.0,8.0,-7.0


## Removing symmetrically equivalent boundaries

Imagine we are considering a modelling project looking at a large range of boundaries and we build a set of grain boundary catalogues corresponding to a variety of axis and misorientation angle combinations. We then concatenate these catalogues into a master catalogue and down select only the subset that contain fewer than a certain number of atoms (perhaps to satisfy computational constraints). Now, because of the high symmetry of the cubic system, this master catalogue will contain multiple entries that actually correspond to equivalent boundaries. A given boundary can typically be specified via multiple different sets of (axis, misorientation angle, boundary plane normal) or, in terms of our parameters, multiple combinations of (`h`, `k`, `l`, `m`, `n`, `H`, `K`, `L`). 

In fact, even for our single (axis, misorientation angle) combination used to build the catalogue above, we will have multiple equivalent boundaries

The `grainboundary` module contains a function `gb_df_dedupe` that will take a catalogue dataframe and return two new dataframes, one from which the duplicate boundaries have been removed and one which is the original dataframe but sorted so that equivalent boundaries are next to each other and flagged in blocks of equivalent cases.

We can try this out on our catalogue here:

In [30]:
gb_df_deduped, gb_df_dedupe_blocks = grainboundary.dedupe_catalogue(gb_df, 'fcc')

The catalogue without duplicates is rather smaller than the original

In [31]:
gb_df_deduped

,h,k,l,m,n,theta,sigma,H,K,L,num_atoms,type,N_b[0],N_b[1],N_b[2],N_w[0],N_w[1],N_w[2],block
2,1,1,1,1,5,38.213211,7.0,-1.0,-2.0,1.0,28.0,symmetric,-2.0,-1.0,3.0,-3.0,1.0,2.0,1
0,1,1,1,1,5,38.213211,7.0,1.0,-2.0,2.0,28.0,mixed,-1.0,3.0,5.0,-1.0,5.0,3.0,2
1,1,1,1,1,5,38.213211,7.0,-2.0,1.0,0.0,28.0,mixed,-2.0,-1.0,-4.0,-1.0,-2.0,-4.0,3
5,1,1,1,1,5,38.213211,7.0,2.0,1.0,2.0,28.0,mixed,0.0,7.0,0.0,3.0,6.0,-2.0,4
27,1,1,1,1,5,38.213211,7.0,2.0,3.0,-3.0,56.0,symmetric,5.0,-1.0,-4.0,5.0,-4.0,-1.0,5
26,1,1,1,1,5,38.213211,7.0,2.0,-3.0,-2.0,56.0,mixed,4.0,-5.0,8.0,-1.0,-2.0,10.0,6
28,1,1,1,1,5,38.213211,7.0,-3.0,-3.0,-1.0,56.0,mixed,-2.0,-8.0,3.0,-6.0,-5.0,4.0,7
30,1,1,1,1,5,38.213211,7.0,3.0,-3.0,2.0,56.0,mixed,1.0,4.0,9.0,0.0,7.0,7.0,8
46,1,1,1,1,5,38.213211,7.0,-1.0,0.0,0.0,84.0,twist,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,9
42,1,1,1,1,5,38.213211,7.0,0.0,1.0,-1.0,84.0,mixed,1.0,-1.0,-2.0,1.0,-2.0,-1.0,10
